In [1]:
from pathlib import Path
import os
import shutil
import cv2
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

In [2]:
images = []

In [3]:
for path in Path("data/extracted_squares/extracted_pieces").rglob("*.jpg"):
    images.append(path)

In [4]:
len(images)

3520

In [6]:
features = []

In [5]:
model = models.mobilenet_v2(pretrained=True)
model.classifier = torch.nn.Identity()  # strip classification head
model.eval()

transform = T.Compose([
    T.Resize((96, 96)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

def extract_embedding(img_path):
    img = Image.open(img_path).convert("RGB")
    x   = transform(img).unsqueeze(0)
    with torch.no_grad():
        return model(x).squeeze().numpy()  # 1280-dim

/Users/hamdan/PycharmProjects/SIChess/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/hamdan/PycharmProjects/SIChess/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/hamdan/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100.0%


In [7]:
for img_path in images:
    feat = extract_embedding(img_path)
    features.append(feat)

In [8]:
features = np.array(features)

In [9]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [10]:
pca = PCA(n_components=50)
features_pca = pca.fit_transform(features_scaled)

In [11]:
kmeans = KMeans(n_clusters=14, n_init=20)
labels = kmeans.fit_predict(features_pca)

In [13]:
import uuid
for path, label in zip(images, labels):
    dest = f"data/clusters/{label}/"
    os.makedirs(dest, exist_ok=True)
    ext      = os.path.splitext(path)[1]
    new_name = f"{Path(path).stem}_{uuid.uuid4().hex[:8]}{ext}"
    shutil.copy(path, os.path.join(dest, new_name))